# Notebook 02: Duplicates, grouping and partitions

**Framework component:** F2, data preparation, steps 4 to 8 of the data-preparation procedure. Needs the outputs of Notebook 01 in Google Drive.

This notebook:

1. removes exact duplicate files (same SHA-256), keeping one copy;
2. finds near-duplicate candidates with a perceptual-hash Hamming distance of 8 or less (protocol v2);
3. quarantines, at a distance of 6 or less, (a) any Ghanaian (D3) image that is a copy of a development image and (b) both images of a pair with different research labels;
4. joins same-label near-duplicates (distance 8 or less) into groups, so that a group never crosses partitions (the plant identifiers are missing in every source, so this is the proxy the protocol allows);
5. splits D1, D4 and D6 by group and class with seed 2026 into training 70%, selection 10%, temperature calibration 5%, threshold selection 5% and internal test 10%; D3 stays whole as the external test;
6. saves image sheets of duplicate candidates for inspection, the frozen manifest with its SHA-256, the count tables for Tables 4.2 and 4.3, and the class weights from the training counts.

**Runtime:** CPU is enough. About 10 to 15 minutes. Re-running produces the same result (fixed seed).

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, tarfile, hashlib, datetime, platform, collections, random, zipfile
import pandas as pd, numpy as np
DRIVE = '/content/drive/MyDrive/ocdf'
WORK = '/content/working'
OUT = f'{DRIVE}/records/nb02'; os.makedirs(OUT, exist_ok=True); os.makedirs(f'{OUT}/sheets', exist_ok=True)
PROTOCOL_VERSION = 'v2'
SEED = 2026
GROUP_MAX_DIST = 8        # v2: pairs at d <= 8 with the same label are kept in one group
QUARANTINE_MAX_DIST = 6   # v2: label conflicts and development-vs-D3 copies are quarantined only at d <= 6
# v1 used 5 for both. Changed after inspecting the v1 sheets, before any training (see protocol record).
inv = pd.read_csv(f'{DRIVE}/manifests/inventory_all_v1.csv', dtype=str, keep_default_na=False)
inv['sample_id'] = [f'{s}-{i:06d}' for i, s in enumerate(inv.source_id)]
print(len(inv), 'inventory rows')
env = {'notebook': '02_duplicates_grouping_partitions', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
       'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__, 'platform': platform.platform()}

In [ ]:
# 2. Helper functions: near-duplicate search, union-find, grouped split, class weights
# ---- OCDF F2 duplicate grouping and partitioning helpers (Notebook 02) ----
import random, collections
import numpy as np

DEV_SOURCES = ('D1', 'D4', 'D6')
EXT_SOURCES = ('D3',)
PARTITIONS = [('train', 0.70), ('select', 0.10), ('calib_T', 0.05), ('thresh', 0.05), ('test', 0.10)]
SOURCE_PRIORITY = {'D1': 0, 'D4': 1, 'D6': 2, 'D3': 3}

class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)

def near_duplicate_pairs(hashes, max_dist=8, chunk=1500):
    """All index pairs (a < b) with Hamming distance <= max_dist between 64-bit hex hashes.

    Exhaustive comparison in chunks with NumPy's bitwise_count (NumPy 2.0 or later).
    """
    H = np.array([int(h, 16) for h in hashes], dtype=np.uint64)
    pairs = []
    for i0 in range(0, len(H), chunk):
        blk = H[i0:i0 + chunk]
        d = np.bitwise_count(blk[:, None] ^ H[None, :])
        ii, jj = np.nonzero(d <= max_dist)
        for a, b in zip(ii + i0, jj):
            if a < b: pairs.append((int(a), int(b), int(d[a - i0, b])))
    return pairs

def grouped_split(groups_by_stratum, seed=2026, partitions=PARTITIONS):
    """Assign whole groups to partitions, per stratum, approximating the target shares by image count.

    groups_by_stratum: {stratum: [(group_id, n_images), ...]}. Returns {group_id: partition}.
    Groups are shuffled with the seed, then placed largest first into the partition whose share is
    furthest below its target. Group integrity always wins over exact shares.
    """
    rng = random.Random(seed)
    out = {}
    names = [p for p, _ in partitions]; target = dict(partitions)
    for stratum in sorted(groups_by_stratum):
        gs = list(groups_by_stratum[stratum]); rng.shuffle(gs)
        gs.sort(key=lambda g: -g[1])                     # stable: equal sizes keep shuffled order
        total = sum(n for _, n in gs); filled = {p: 0 for p in names}
        for gid, n in gs:
            # deficit after placing: choose partition with the largest remaining shortfall
            best = max(names, key=lambda p: (target[p] * total - filled[p]) / max(target[p], 1e-9))
            out[gid] = best; filled[best] += n
    return out

def class_weights(train_counts, cap=5.0):
    """Training recipe: inverse-frequency weights normalised as n / (k * n_c) (so the image-weighted mean is 1),
    then capped at `cap`."""
    labels = sorted(train_counts); n = sum(train_counts.values()); k = len(labels)
    return {c: round(min(n / (k * train_counts[c]), cap), 4) for c in labels}

In [ ]:
# 3. Starting pool and first exclusions (unreadable files, labels not selected)
excl = []   # rows: sample_id, source_id, original_label, research_label, reason, detail
def exclude(df, reason, detail=''):
    for r in df.itertuples():
        excl.append({'sample_id': r.sample_id, 'source_id': r.source_id, 'original_label': r.original_label,
                     'research_label': r.research_label, 'reason': reason, 'detail': detail if isinstance(detail, str) else detail(r)})
exclude(inv[inv.decode_ok != 'True'], 'unreadable', lambda r: r.decode_error[:120])
exclude(inv[(inv.decode_ok == 'True') & (inv.selected != 'True')], 'label not selected', lambda r: r.exclusion_reason)
pool = inv[(inv.decode_ok == 'True') & (inv.selected == 'True')].copy()
print('readable images with a selected label:', len(pool))
print(pool.groupby(['source_id', 'research_label']).size().unstack(fill_value=0))

In [ ]:
# 4. Exact duplicates (step 4, first part)
pool['prio'] = pool.source_id.map(SOURCE_PRIORITY)
pool = pool.sort_values(['sha256', 'prio', 'original_relpath'])
quarantine = {}   # sample_id -> reason
drop_dup = []
for sha, g in pool.groupby('sha256', sort=False):
    if len(g) == 1: continue
    labels = set(g.research_label); srcs = set(g.source_id)
    if len(labels) > 1:                                   # same file, different labels: ambiguous
        for s in g.sample_id: quarantine[s] = f'exact duplicate with conflicting labels {sorted(labels)}'
        continue
    if srcs & set(DEV_SOURCES) and srcs & set(EXT_SOURCES):  # same file in development and Ghana
        for r in g[g.source_id.isin(EXT_SOURCES)].itertuples(): quarantine[r.sample_id] = 'exact copy of a development image'
        g = g[~g.source_id.isin(EXT_SOURCES)]
    keep = g.iloc[0]
    for r in g.iloc[1:].itertuples(): drop_dup.append((r.sample_id, keep.sample_id))
dup_ids = {a for a, _ in drop_dup}
exclude(pool[pool.sample_id.isin(dup_ids)], 'exact duplicate removed', lambda r: 'copy of ' + dict(drop_dup)[r.sample_id])
uniq = pool[~pool.sample_id.isin(dup_ids)].reset_index(drop=True)
print('exact duplicate copies removed:', len(dup_ids), '| unique images:', len(uniq))

In [ ]:
# 5. Near-duplicate candidates (step 4, second part), quarantine (steps 6 and 7) and groups (step 5)
pairs = near_duplicate_pairs(list(uniq.phash), GROUP_MAX_DIST)
S = uniq.source_id.values; L = uniq.research_label.values; ID = uniq.sample_id.values
def ptype(a, b):
    sa, sb = S[a], S[b]
    if sa == sb: return 'within ' + sa
    if (sa in DEV_SOURCES) and (sb in DEV_SOURCES): return 'development cross-source'
    if (sa in EXT_SOURCES) and (sb in EXT_SOURCES): return 'within external'
    return 'development vs D3'
rows = []
for a, b, d in pairs:
    rows.append({'a': ID[a], 'b': ID[b], 'dist': d, 'type': ptype(a, b), 'label_a': L[a], 'label_b': L[b],
                 'label_conflict': L[a] != L[b], 'ia': a, 'ib': b})
P = pd.DataFrame(rows, columns=['a', 'b', 'dist', 'type', 'label_a', 'label_b', 'label_conflict', 'ia', 'ib'])
for r in P[P.dist <= QUARANTINE_MAX_DIST].itertuples():
    if r.label_conflict:
        quarantine.setdefault(r.a, f'near duplicate (d={r.dist}) of an image with a different label')
        quarantine.setdefault(r.b, f'near duplicate (d={r.dist}) of an image with a different label')
    elif r.type == 'development vs D3':
        ext = r.a if S[r.ia] in EXT_SOURCES else r.b
        quarantine.setdefault(ext, f'near duplicate (d={r.dist}) of a development image')
exclude(uniq[uniq.sample_id.isin(quarantine)], 'quarantined', lambda r: quarantine[r.sample_id])
keep_mask = ~uniq.sample_id.isin(quarantine)
uf = UnionFind(len(uniq))
for r in P[(~P.label_conflict) & (P.type != 'development vs D3')].itertuples():   # label-pure; never across dev/external
    if keep_mask[r.ia] and keep_mask[r.ib]: uf.union(r.ia, r.ib)
uniq['group_root'] = [uf.find(i) for i in range(len(uniq))]
ret = uniq[keep_mask].copy()
# stable, readable group ids
roots = {rt: f'G{k:06d}' for k, rt in enumerate(sorted(ret.group_root.unique()))}
ret['group_id'] = ret.group_root.map(roots)
assert ret.groupby('group_id').research_label.nunique().max() == 1, 'a group contains two labels'
assert not ((ret.groupby('group_id').source_id.agg(lambda x: bool(set(x) & set(EXT_SOURCES)) and bool(set(x) & set(DEV_SOURCES))))).any(), 'a group crosses development and external data'
gsize = ret.groupby('group_id').size()
nd_stats = {
  'pairs_total': int(len(P)),
  'pairs_by_type': P.groupby('type').size().to_dict(),
  'pairs_by_distance': {int(k): int(v) for k, v in P.groupby('dist').size().items()},
  'label_conflict_pairs': int(P.label_conflict.sum()),
  'label_conflict_pairs_by_distance': {int(k): int(v) for k, v in P[P.label_conflict].groupby('dist').size().items()},
  'pairs_by_type_and_distance': {f'{t} d={d}': int(n) for (t, d), n in P.groupby(['type', 'dist']).size().items()},
  'quarantined_images': len(quarantine),
  'quarantine_reasons': collections.Counter(v.split(' (')[0].split(' [')[0] for v in quarantine.values()),
  'groups': int(len(gsize)), 'groups_with_more_than_one_image': int((gsize > 1).sum()),
  'group_size_distribution': {int(k): int(v) for k, v in gsize.value_counts().sort_index().items()},
  'largest_groups': [{'group_id': g, 'size': int(n), 'source': ','.join(sorted(set(ret[ret.group_id == g].source_id))),
                      'label': ret[ret.group_id == g].research_label.iloc[0]} for g, n in gsize.sort_values(ascending=False).head(10).items()],
}
print(json.dumps(nd_stats, indent=1, default=int))

In [ ]:
# 5b. Diagnostic: how close is each image to its nearest neighbour in the same source? (informs the threshold)
# pHash sets exactly 32 of 64 bits, so distances between two hashes are always even.
H = np.array([int(h, 16) for h in uniq.phash], dtype=np.uint64)
rng = np.random.default_rng(SEED); nn_rows = []; nn_dist = {}
for src in sorted(set(S)):
    idx = np.where(S == src)[0]
    smp = rng.choice(idx, size=min(3000, len(idx)), replace=False)
    d_all = []
    for i in smp:
        d = np.bitwise_count(H[idx] ^ H[i]).astype(int); d[idx == i] = 99
        j = idx[int(np.argmin(d))]; dm = int(d.min()); d_all.append(dm)
        if 6 <= dm <= 10: nn_rows.append({'a': ID[i], 'b': ID[j], 'dist': dm, 'label_a': L[i], 'label_b': L[j], 'source': src})
    c = collections.Counter(d_all)
    nn_dist[src] = {'sampled_images': int(len(smp)), 'share_with_neighbour_at_d<=4': round(sum(v for k, v in c.items() if k <= 4) / len(smp), 4),
                    'share_d6_to_10': round(sum(v for k, v in c.items() if 6 <= k <= 10) / len(smp), 4),
                    'share_d12_to_16': round(sum(v for k, v in c.items() if 12 <= k <= 16) / len(smp), 4),
                    'median_nearest_distance': float(np.median(d_all))}
NN = pd.DataFrame(nn_rows, columns=['a', 'b', 'dist', 'label_a', 'label_b', 'source'])
nd_stats['nearest_neighbour_diagnostic'] = nn_dist
print(json.dumps(nn_dist, indent=1))

In [ ]:
# 6. Image sheets of candidates, for inspection (step 4: inspect before deciding)
from PIL import Image, ImageDraw
for t in glob.glob(f'{DRIVE}/working/working_*.tar'):
    src = os.path.basename(t)[8:-4]
    if not os.path.isdir(f'{WORK}/{src}'):
        with tarfile.open(t) as tar: tar.extractall(WORK, filter='data')
wp = dict(zip(uniq.sample_id, uniq.working_path))
def thumb(sid, size=180):
    im = Image.open(f'{WORK}/{wp[sid]}').convert('RGB'); im.thumbnail((size, size)); c = Image.new('RGB', (size, size), 'white')
    c.paste(im, ((size - im.width) // 2, (size - im.height) // 2)); return c
def pair_sheet(df, title, path, n=16, size=180):
    df = df.sample(min(n, len(df)), random_state=SEED) if len(df) else df
    if not len(df): return None
    cols = 4; rws = (len(df) + cols - 1) // cols; W = cols * (2 * size + 20); H = 40 + rws * (size + 40)
    sheet = Image.new('RGB', (W, H), 'white'); dr = ImageDraw.Draw(sheet); dr.text((10, 10), title, fill='black')
    for k, r in enumerate(df.itertuples()):
        x = (k % cols) * (2 * size + 20); y = 40 + (k // cols) * (size + 40)
        sheet.paste(thumb(r.a, size), (x, y)); sheet.paste(thumb(r.b, size), (x + size, y))
        dr.text((x, y + size + 4), f'd={r.dist} {r.a} {r.label_a} | {r.b} {r.label_b}'[:60], fill='black')
    sheet.save(path); return path
sheets = []
for name, sub in [('within_source_d0-4', P[(~P.type.str.contains('vs')) & (P.dist <= 4)]),
                  ('within_source_d6', P[(~P.type.str.contains('vs')) & (P.dist == 6)]),
                  ('within_source_d8', P[(~P.type.str.contains('vs')) & (P.dist == 8)]),
                  ('development_vs_D3', P[P.type == 'development vs D3']),
                  ('label_conflicts_quarantined_d<=6', P[P.label_conflict & (P.dist <= QUARANTINE_MAX_DIST)]),
                  ('label_conflicts_not_quarantined_d8', P[P.label_conflict & (P.dist > QUARANTINE_MAX_DIST)])] + \
                 [(f'nearest_d6-10_{src}', NN[NN.source == src]) for src in sorted(set(NN.source))]:
    p = pair_sheet(sub, f'{name}: {len(sub)} pairs (random sample of up to 16)', f'{OUT}/sheets/{name}.png')
    if p: sheets.append(p); print('saved', p)
# a sample from the largest group
if len(gsize) and gsize.max() > 1:
    g0 = gsize.idxmax(); ids = list(ret[ret.group_id == g0].sample_id)[:24]; size = 150; cols = 8
    sh = Image.new('RGB', (cols * size, 40 + ((len(ids) + cols - 1) // cols) * size), 'white')
    ImageDraw.Draw(sh).text((10, 10), f'largest group {g0}: {gsize.max()} images (first {len(ids)} shown)', fill='black')
    for k, s in enumerate(ids): sh.paste(thumb(s, size), ((k % cols) * size, 40 + (k // cols) * size))
    p = f'{OUT}/sheets/largest_group.png'; sh.save(p); sheets.append(p); print('saved', p)

In [ ]:
# 7. Grouped partitions (step 5 onwards): D1, D4, D6 by group and class with seed 2026; D3 external
dev = ret[ret.source_id.isin(DEV_SOURCES)]
g_info = dev.groupby('group_id').agg(n=('sample_id', 'size'), label=('research_label', 'first'),
                                     srcs=('source_id', lambda s: '+'.join(sorted(set(s)))))
strata = collections.defaultdict(list)
for gid, r in g_info.iterrows(): strata[(r.srcs, r.label)].append((gid, int(r.n)))
assign = grouped_split(strata, seed=SEED)
ret['partition'] = [assign.get(g, 'external') if s in DEV_SOURCES else 'external' for g, s in zip(ret.group_id, ret.source_id)]
assert set(ret[ret.source_id.isin(EXT_SOURCES)].partition) == {'external'}
order = ['train', 'select', 'calib_T', 'thresh', 'test', 'external']
part_counts = ret.pivot_table(index=['source_id', 'research_label'], columns='partition', values='sample_id', aggfunc='count', fill_value=0)[[c for c in order if c in set(ret.partition)]]
part_groups = ret.pivot_table(index=['source_id', 'research_label'], columns='partition', values='group_id', aggfunc='nunique', fill_value=0)[[c for c in order if c in set(ret.partition)]]
print('Images per partition:'); print(part_counts); print('Groups per partition:'); print(part_groups)
tg = part_groups['test'] if 'test' in part_groups else pd.Series(dtype=int)
tg = tg[[i for i in tg.index if i[0] in DEV_SOURCES]]; limited = tg[tg < 30]
print('Strata with fewer than 30 groups in the internal test (estimates to be labelled limited):', limited.to_dict())

In [ ]:
# 8. Reconciliation table , class weights, frozen manifest and protocol record (step 8)
E = pd.DataFrame(excl)
rec = inv.groupby(['source_id', 'original_label']).agg(research_label=('research_label', 'first'), downloaded=('sample_id', 'size'))
for reason in ['unreadable', 'label not selected', 'exact duplicate removed', 'quarantined']:
    c = E[E.reason == reason].groupby(['source_id', 'original_label']).size()
    rec[reason] = c.reindex(rec.index, fill_value=0)
rec['retained'] = ret.groupby(['source_id', 'original_label']).size().reindex(rec.index, fill_value=0)
assert (rec.downloaded == rec[['unreadable', 'label not selected', 'exact duplicate removed', 'quarantined', 'retained']].sum(axis=1)).all(), 'counts do not reconcile'
print(rec)

train = ret[ret.partition == 'train']
weights = {crop: class_weights(train[train.crop == crop].research_label.value_counts().to_dict())
           for crop in ['cassava', 'maize']}
print('class weights (training counts):', weights)

cols = ['sample_id', 'source_id', 'crop', 'original_label', 'research_label', 'partition', 'group_id',
        'working_path', 'sha256', 'phash', 'width', 'height', 'original_relpath']
manifest = ret[cols].sort_values(['crop', 'partition', 'sample_id'])
mpath = f'{DRIVE}/manifests/manifest_{PROTOCOL_VERSION}.csv'
manifest.to_csv(mpath, index=False)
E.to_csv(f'{DRIVE}/manifests/exclusions_{PROTOCOL_VERSION}.csv', index=False)
P.drop(columns=['ia', 'ib']).to_csv(f'{OUT}/near_duplicate_pairs.csv', index=False)
rec.to_csv(f'{OUT}/table_4_2_reconciliation.csv'); part_counts.to_csv(f'{OUT}/table_4_3_partition_images.csv')
part_groups.to_csv(f'{OUT}/table_4_3_partition_groups.csv')
msha = hashlib.sha256(open(mpath, 'rb').read()).hexdigest()
protocol = {'protocol_version': PROTOCOL_VERSION, 'frozen_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'manifest': os.path.basename(mpath), 'manifest_sha256': msha, 'manifest_rows': int(len(manifest)),
            'seed': SEED, 'partitions': dict(PARTITIONS), 'phash': '64-bit DCT pHash (imagehash method)',
            'group_max_hamming': GROUP_MAX_DIST, 'quarantine_max_hamming': QUARANTINE_MAX_DIST,
            'change_from_v1': ('v1 used d <= 5 for grouping and quarantine. Inspection of the v1 sheets showed the same photograph '
                               'at d = 6 (D1, Healthy vs CBB) and repeated PlantVillage leaves at d = 6 to 8, while d = 8 also matched '
                               'unrelated images (cassava vs maize). v2 groups same-label pairs at d <= 8 and quarantines only at d <= 6. '
                               'Changed before any model training.'), 'exact_duplicate_rule': 'keep one copy; priority D1, D4, D6, D3, then path',
            'quarantine_rules': ['D3 image that is an exact or near copy of a development image',
                                 'both images of a near-duplicate pair with different research labels',
                                 'all copies of an exact duplicate with different research labels'],
            'grouping': 'union of near-duplicate pairs (d <= 5) among retained images; no plant/farm/session identifiers exist in any source',
            'class_weights_from_training_counts': weights, 'limited_test_strata': {str(k): int(v) for k, v in limited.items()},
            'environment': env}
json.dump(protocol, open(f'{DRIVE}/records/protocol_{PROTOCOL_VERSION}.json', 'w'), indent=1, default=int)
json.dump(nd_stats, open(f'{OUT}/near_duplicate_stats.json', 'w'), indent=1, default=int)
print('manifest frozen:', mpath, msha)

res = '/content/nb02_results.zip'
with zipfile.ZipFile(res, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [f'{DRIVE}/records/protocol_{PROTOCOL_VERSION}.json', f'{OUT}/near_duplicate_stats.json', f'{OUT}/near_duplicate_pairs.csv',
              f'{OUT}/table_4_2_reconciliation.csv', f'{OUT}/table_4_3_partition_images.csv', f'{OUT}/table_4_3_partition_groups.csv',
              f'{DRIVE}/manifests/exclusions_{PROTOCOL_VERSION}.csv', mpath] + sheets:
        z.write(p, os.path.basename(p))
from google.colab import files
files.download(res)

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb02_results.zip`. The copies used for the published results are in this repository under `records/nb02/`.